# Linear Regression and Correlation

**Estimated Time**: 25-30 Minutes  
**Professor**: Elizabeth Schwartz </br>
**Developers**: Ariana Ghimire, Rinrada Maneenop  </br>
**Textbook Reference**: OpenStax *Introductory Statistics 2e*, Chapter 12 (Linear Regression and Correlation)  
**Goal**: Learn the basics of scatter plots, correlation, and simple linear regression with guided practice.

## Table of Contents

1. Quick Concepts and Setup  
2. Load and View the Data  
3. Scatter Plot (Always Start Here)  
4. Correlation Coefficient `r`  
5. Regression Line `y_hat = a + bx`  
6. Prediction and Caution About Extrapolation  
7. Short Practice Exercises

## Notebook Structure

You will see short **example -> practice** cycles.

For practice cells, replace each `...` with your code.  
Then run the cell to check your result.

This notebook follows OpenStax Ch. 12 guidance:
- start with a scatter plot,
- use `r` to describe strength/direction,
- use the regression line for prediction only when linear and within the observed x-range.

---

## 1. Introduction

### 1.1. Learning Objectives

Do NBA players who score more points earn higher salaries? Are there other stats that might predict pay even better? In this notebook, we'll use **real NBA player data** to explore these questions using the tools of **linear regression and correlation**. By the end, you will:

- Understand what a **scatter plot** tells us about the relationship between two variables
- Measure the strength of a linear relationship using the **correlation coefficient *r***
- Calculate and interpret a **regression line** (line of best fit)
- Use the regression line to make **predictions**
- Apply these skills to explore a relationship of your own choosing in the sandbox

### 1.2. What is Linear Regression?

Suppose you want to predict an NBA player's salary based on how many points they score per game. You have data on hundreds of players, but there's no perfect rule — a player who scores 20 points a game doesn't always earn exactly twice as much as one who scores 10. Instead, the data follows a **trend**.

**Linear Regression** is a method that finds the straight line that best describes this trend. The line has the form:

> **ŷ = a + b · x**

Where:
- **x** is the *independent variable* (the input — e.g., points per game)
- **ŷ** ("y-hat") is the *predicted value* of the *dependent variable* (the output — e.g., salary)
- **b** is the *slope* — how much ŷ changes for each 1-unit increase in x
- **a** is the *y-intercept* — the value of ŷ when x equals 0

Before fitting a line, we also measure **correlation** — a number between -1 and 1 that describes how strongly and in which direction two variables move together.

### 1.3. Setup

Below, we have imported the Python libraries needed for this module. Run the code in this cell before running any other code cells, and be careful **not to change** any of the code.

You can run the cell in any of these ways:
- **Ctrl + Enter**: Run the cell and keep the cursor in the same cell.
- **Shift + Enter**: Run the cell and move the cursor to the next cell.
- **Click the Play button**: Click the Run (play) button to the left of the cell to execute it.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

from visuals import (
    show_interactive_correlation,
    show_interactive_line_tuner,
    show_prediction_slider,
    show_sandbox_regression,
)

# Make plots look clean
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

---

## 2. Data Preparation

### 2.1. Loading the Data

We will work with an **NBA player dataset** from the Data 8 Spring 2025 course materials. It contains statistics and salary information for NBA players across a season. Let's load it and take a first look!

In [ ]:
# Load the dataset from the Data 8 SP25 review sandbox
url = "https://raw.githubusercontent.com/data-8/materials-sp25/main/review-sandbox/nba.csv"
nba = pd.read_csv(url)

# Display the first 10 rows
display(nba.head(10))
print(f"Shape: {nba.shape[0]} rows x {nba.shape[1]} columns")
print(f"Columns: {list(nba.columns)}")

### 2.2. Exploring the Data

Before we do any analysis, let's get a feel for the numbers. The cell below prints summary statistics — the mean, standard deviation, minimum, and maximum — for all numeric columns.

In [ ]:
# Summary statistics for numeric columns
display(nba.describe().round(2))

**Question 2.2.** Look at the summary statistics for `Salary`. What is the average salary? What is the range (min to max)? Does the spread surprise you — and why or why not?

*Your Answer Here*

### 2.3. Cleaning the Data

Real-world data often has missing values. Let's check whether any rows are missing `Salary` or `Points`, and drop those rows so our analysis is based on complete records only.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-8/materials-sp25/main/review-sandbox/nba.csv"
nba = pd.read_csv(url)

print("Missing values per column:")
print(nba.isnull().sum())
print()

nba_clean = nba.dropna(subset=["Salary", "Points"]).copy()
print(f"Rows before cleaning: {len(nba)}")
print(f"Rows after cleaning:  {len(nba_clean)}")

**Question 2.3.** Why is it important to remove rows with missing `Salary` or `Points` values before fitting a regression line? What might happen to our analysis if we left them in?

*Your Answer Here*

---

## 3. Visualizing the Data

### 3.1. Scatter Plots

A **scatter plot** is always the first step when studying the relationship between two numerical variables. Each dot below represents one NBA player — the x-axis shows points per game (`Points`) and the y-axis shows salary.

When looking at a scatter plot, ask yourself:
- Does the cloud of points slope **upward** (positive), **downward** (negative), or show **no pattern** (no relationship)?
- How tightly **clustered** are the points around an imaginary line?
- Are there any **outliers** that sit far from the rest?

In [ ]:
fig, ax = plt.subplots()

ax.scatter(nba_clean["Points"], nba_clean["Salary"],
           alpha=0.45, color="steelblue", edgecolors="white", s=65)

ax.set_xlabel("Points Per Game", fontsize=12)
ax.set_ylabel("Salary", fontsize=12)
ax.set_title("NBA Players: Points Per Game vs. Salary", fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"${val/1e6:.1f}M"))

plt.tight_layout()
plt.show()

**Question 3.1.** Describe what you see in the scatter plot. Is there a clear direction to the relationship? Does it look linear? Would you say the relationship is strong or weak — and why?

*Your Answer Here*

**Question 3.2.** Do you notice any outliers — players who seem to earn much more than their scoring average would predict? What might explain why a player commands a very high salary without being a top scorer?

*Your Answer Here*

---

## 4. The Correlation Coefficient

### 4.1. What Does *r* Tell Us?

The **correlation coefficient**, written as ***r***, is a single number between **-1 and +1** that captures the strength and direction of the linear relationship between two variables.

| Value of *r* | Interpretation |
|---|---|
| Close to **+1** | Strong **positive** relationship — as x increases, y tends to increase |
| Close to **-1** | Strong **negative** relationship — as x increases, y tends to decrease |
| Close to **0** | Little to no linear relationship |
| Exactly **+1** or **-1** | Perfect linear relationship (all points on one line) |

Use the interactive visual below to build intuition:
- Move the `Target r` slider to change direction/strength.
- Click **Generate New Sample** to see that random samples vary, even with the same target correlation.

In [ ]:
show_interactive_correlation()

### 4.2. Computing *r*

Now let's compute *r* for the relationship between `Points` and `Salary`. We'll use software to get both *r* and a **p-value**. The p-value tells us whether the correlation is statistically significant — i.e., whether it's unlikely to have occurred by random chance alone. A p-value below 0.05 is typically considered significant.

In [ ]:
x = nba_clean["Points"]
y = nba_clean["Salary"]

r, p_value = stats.pearsonr(x, y)

print(f"Correlation coefficient (r): {r:.4f}")
print(f"p-value:                     {p_value:.2e}")

**Question 4.1 (Code Practice).** Fill in the `...` in the next code cell to extract and interpret the correlation for `Points` vs `Salary`.

In [ ]:
# Fill in the blanks to interpret the correlation result
r_value = ...  # use the value already computed above

if r_value > 0:
    direction = ...  # "positive" or "negative"
else:
    direction = ...

print(f"r = {r_value:.3f}")
print(f"Direction: {direction}")

**Question 4.2 (Code Practice).** Fill in the `...` in the next code cell to test whether the correlation is statistically significant at `alpha = 0.05`.

<details>
<summary><b>HINTS: See our staff solution outline for Exercise 4.1</b></summary>

```python
# Solution: Exercise 4.1

r_value = r

if r_value > 0:
    direction = "positive"
else:
    direction = "negative"

print(f"r = {r_value:.3f}")
print(f"Direction: {direction}")
```
</details>

In [ ]:
# Fill in the blanks for a simple significance decision
alpha = 0.05
is_significant = ...  # compare p_value with alpha

print(f"p-value = {p_value:.2e}")
print(f"Significant at alpha={alpha}? {is_significant}")

if is_significant:
    print("There is evidence of a linear relationship between Points and Salary.")
else:
    print("There is not enough evidence of a linear relationship.")

---

<details>
<summary><b>HINTS: See our staff solution outline for Exercise 4.2</b></summary>

```python
# Solution: Exercise 4.2

alpha = 0.05
is_significant = p_value < alpha

print(f"p-value = {p_value:.2e}")
print(f"Significant at alpha={alpha}? {is_significant}")

if is_significant:
    print("There is evidence of a linear relationship between Points and Salary.")
else:
    print("There is not enough evidence of a linear relationship.")
```
</details>

## 5. The Regression Line

### 5.1. Slope and Intercept

Since there's a meaningful positive correlation, we can fit a **regression line** — the single straight line that minimizes the total squared distance between itself and all the data points. This is called the **Least Squares Regression Line**.

The formulas for slope and intercept are:

$$
b = r\left(\frac{s_y}{s_x}\right)
$$

- $b$: slope of the regression line (how much predicted salary changes for +1 point)
- $r$: correlation coefficient between $x$ and $y$
- $s_y$: standard deviation of $y$ (here, `Salary`)
- $s_x$: standard deviation of $x$ (here, `Points`)

$$
a = \bar{y} - b\bar{x}
$$

- $a$: intercept of the regression line (predicted salary when points = 0)
- $\bar{y}$: mean of $y$
- $\bar{x}$: mean of $x$
- $x$: observed value of the independent variable
- $\hat{y}$: predicted value from the regression line

Let's compute these step by step using the Chapter 12 formulas.

In [ ]:
# Step 1: Compute means and standard deviations
x_mean, y_mean = x.mean(), y.mean()
x_std,  y_std  = x.std(),  y.std()

print(f"Mean Points : {x_mean:.3f} points/game")
print(f"Mean Salary : ${y_mean:,.0f}")
print(f"Std  Points : {x_std:.3f}")
print(f"Std  Salary : ${y_std:,.0f}")

In [ ]:
# Step 2: Compute slope (b) and intercept (a)
b = r * (y_std / x_std)
a = y_mean - b * x_mean

print(f"Slope     (b) : ${b:,.0f} per additional point/game")
print(f"Intercept (a) : ${a:,.0f}")
print(f"\nRegression equation: y_hat = {a:,.0f} + {b:,.0f} * x")

In [ ]:
# Textbook check: the least-squares line passes through (x_mean, y_mean)
y_hat_at_x_mean = a + b * x_mean
print(f"y_mean           : ${y_mean:,.0f}")
print(f"y_hat at x_mean  : ${y_hat_at_x_mean:,.0f}")

### 5.2. Drawing the Best-Fit Line

Let's now draw the regression line on top of our scatter plot to see how well it fits the data.

In [ ]:
x_range = np.linspace(x.min(), x.max(), 200)
y_hat   = a + b * x_range

fig, ax = plt.subplots()

ax.scatter(x, y, alpha=0.35, color="steelblue",
           edgecolors="white", s=60, label="Players")
ax.plot(x_range, y_hat, color="crimson", linewidth=2.5,
        label=f"y_hat = {a/1e6:.2f}M + {b/1e6:.2f}M * x  (r = {r:.3f})")

ax.set_xlabel("Points Per Game", fontsize=12)
ax.set_ylabel("Salary", fontsize=12)
ax.set_title("NBA Players: Regression Line (Points vs. Salary)", fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"${val/1e6:.1f}M"))
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### 5.2.1 Interactive: Try Your Own Line

OpenStax emphasizes that the least-squares line is the line that makes the total squared error as small as possible.

Use the sliders below to choose your own slope and intercept. Compare your line to the best-fit line and watch the **SSE** (sum of squared errors).

- Smaller SSE = better fit
- The best-fit line should usually have one of the smallest SSE values

In [ ]:
show_interactive_line_tuner(x=x, y=y, a=a, b=b, x_range=x_range, y_hat=y_hat)

**Question 5.1 (Code Practice).** Fill in the `...` in the next code cell to interpret the slope in context.

<details>
<summary><b>HINTS: See our staff solution outline for Exercise 5.1</b></summary>

```python
# Solution: Exercise 5.1

salary_change_per_point = b
points_increase = 1
predicted_change = salary_change_per_point * points_increase

print(f"For each +{points_increase} point/game, predicted salary changes by ${predicted_change:,.0f}.")
```
</details>

In [ ]:
# Fill in the blanks to interpret slope b
salary_change_per_point = ...  # use b
points_increase = 1
predicted_change = ...  # multiply slope by points increase

print(f"For each +{points_increase} point/game, predicted salary changes by ${predicted_change:,.0f}.")

### 5.3. Making Predictions

One of the most useful things about the regression line is that we can plug in any value of `Points` to get a **predicted salary**. Use the slider below to explore predictions for different scoring averages!

> ⚠️ **Important:** Predictions are only reliable **within the range of the data**. Predicting far outside this range — called **extrapolation** — can lead to unreliable results.

In [ ]:
show_prediction_slider(x=x, y=y, a=a, b=b, x_range=x_range, y_hat=y_hat)

**Question 5.2.** Use the slider to find the predicted salary for a player who scores **20 points per game**. Then verify your answer by completing the code cell below using the regression equation `a + b * x`. Do both answers match?

<details>
<summary><b>HINTS: See our staff solution outline for Exercise 5.2</b></summary>

```python
# Solution: Exercise 5.2

pts_input = 20
predicted_salary = a + b * pts_input

print(f"Predicted salary for {pts_input} points/game: ${predicted_salary/1e6:.2f}M")
```
</details>

In [ ]:
# TODO: Compute the predicted salary for a player who scores 20 points per game.
# Replace the ... with the correct calculation using a and b.
pts_input        = 20
predicted_salary = ...  # your calculation here

print(f"Predicted salary for {pts_input} points/game: ${predicted_salary/1e6:.2f}M")

*Your Answer Here (does your manual calculation match the slider?)*

---

## 6. Exploring Other Relationships (Sandbox)

We've seen how points per game relates to salary. But the NBA dataset has many other numeric columns — rebounds, assists, steals, blocks, and more. Does a different stat predict salary even better?

In this sandbox section, you'll pick **any two numeric columns** and run your own regression analysis using the dropdowns below.

### 6.1. Choosing Variables

In [ ]:
# First, let's remind ourselves of all available numeric columns
numeric_cols = nba_clean.select_dtypes(include="number").columns.tolist()
print("Available numeric columns:")
print(numeric_cols)

In [ ]:
# Choose your x and y variables using the dropdowns, then click Run Regression
show_sandbox_regression(nba_clean)

**Question 6.1.** Which pair of variables did you choose to explore? Before running the regression, what did you expect — a positive relationship, a negative relationship, or no relationship? Explain your intuition.

*Your Answer Here*

### 6.2. Running Your Own Regression

**Question 6.2.** What did you actually find? Report the correlation coefficient *r* for your chosen pair. Is the relationship stronger or weaker than the `Points vs. Salary` relationship from Section 5? Does the result match what you expected in Question 6.1?

*Your Answer Here*

---

## 7. Conclusion

In this notebook, we explored the core ideas of **Linear Regression and Correlation** using real NBA player data:

- A **scatter plot** is the essential first step for understanding the relationship between two numeric variables.
- The **correlation coefficient *r*** (between -1 and +1) measures how strongly and in what direction two variables are linearly related.
- The **regression line** `y_hat = a + b*x` is the best-fit line through the data, computed from *r*, the standard deviations, and the means of x and y.
- We can use the regression line to **predict** values of y for a given x — but only reliably within the observed range of the data (no extrapolation!).

These tools form the foundation for more advanced modeling in data science and statistics. Nice work! 🎉

**Want to explore further?** Try looking at whether `MIN` (minutes played) predicts `Points` better than any salary-related variable, or compare how correlations differ across player positions.

## 📋 Post-Notebook Reflection Form

Thank you for completing the notebook! We'd love to hear your thoughts so we can continue improving and creating content that supports your learning.

Please take a few minutes to fill out this short reflection form:

👉 **[Click here to fill out the Reflection Form](#)**

---

### 🧠 Why it matters:
Your feedback helps us understand:
- How clear and helpful the notebook was
- What you learned from the experience
- What topics you'd like to see in the future

This form is anonymous and should take less than 5 minutes to complete.

We appreciate your time and honest input! 💬

---

**Woohoo! You have completed this notebook! 🚀**